# Learning to rank the candidate pool

Every recommender we have built is a **retrieval path** -- a cheap generator that sees one slice of
the signal. Popularity counts reads; BM25 reads words; item-item and MF read who-read-what; the
two-tower and the semantic path read learned taste. Unit 6 blended their scores with **fixed**
weights; Unit 10 fused a sparse and a dense path. Every one of them ranks by a *single* ordering key
we chose by hand.

This unit asks the production question: our paths each see one slice -- what if a model *learned* to
rank a reader's candidate pool? We keep the two-stage shape Unit 10 made concrete -- **cheap retrieval
proposes a pool, an expensive model re-scores only that pool** -- but we replace the hand-tuned
ordering key with a **learned** one: a small PyTorch **reranker** trained on implicit feedback. Along
the way we hit the single most important failure mode in stacked models -- training **leaks** -- in
two forms: an obvious one that makes the reranker *worse* than doing nothing, and a subtle one that
makes it look *better* than it really is. With both closed we measure, honestly, whether a learned
ranker beats a careful fixed blend at all -- and where its real lift comes from.

## The substrate and the six retrieval paths

The Units 1-10 substrate is unchanged: `generated_dir()` locates the seeded slice, and the catalog,
the gzip'd interaction log, the keyword corpus and the GloVe subset load as before. `test` stays
sealed; we only ever score on `val`.

The reranker does not invent a new signal -- it **re-scores the pool the existing paths propose**. So
we fit the six retrieval paths once (seed 0, their shipped configurations) and reuse them as the
reranker's candidate sources and feature producers everywhere below.

In [1]:
import json
from collections import defaultdict

import numpy as np
import pandas as pd
from bookrec import (
    ItemItemRetrievalPath,
    LexicalRetrievalPath,
    MatrixFactorizationPath,
    NeuralRerankerPath,
    PathRegistry,
    PopularityRetrievalPath,
    RerankerModel,
    SemanticEmbeddingRetrievalPath,
    TwoTowerRetrievalPath,
    generated_dir,
    load_catalog,
    load_glove_subset,
    load_keywords,
    run_blended_scoreboard,
)
from bookrec.rerank import family_mask, feature_names, split_profile_labels

K = 10
POOL = 50
PATH_NAMES = ("popularity", "lexical", "item-item", "mf", "semantic", "two-tower")

data_dir = generated_dir()
catalog = load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)
interactions_path = data_dir / "interactions.csv.gz"
keywords = load_keywords(data_dir / "keywords.csv.gz")
glove = load_glove_subset()
cold_readers = set(json.loads((data_dir / "cold_partitions.json").read_text())["cold_readers"])

# Read the interaction log (gzip auto-detected) into plain dict rows the paths and reranker consume.
rows = pd.read_csv(interactions_path).to_dict("records")

# Each reader's train history (the seen set) and their held-out val positives (the relevant set).
seen_by = defaultdict(set)
val_by = defaultdict(set)
for row in rows:
    if int(row["label"]) == 1:
        if row["split"] == "train":
            seen_by[int(row["reader_id"])].add(int(row["item_id"]))
        elif row["split"] == "val":
            val_by[int(row["reader_id"])].add(int(row["item_id"]))


def fit_six(train_rows):
    # Fit the six retrieval paths whose pool and calibrated scores the reranker consumes (seed 0).
    return {
        "popularity": PopularityRetrievalPath().fit(train_rows, catalog=catalog_ids),
        "lexical": LexicalRetrievalPath(keywords).fit(train_rows, catalog=catalog_ids),
        "item-item": ItemItemRetrievalPath().fit(train_rows, catalog=catalog_ids),
        "mf": MatrixFactorizationPath(seed=0).fit(train_rows, catalog=catalog_ids),
        "semantic": SemanticEmbeddingRetrievalPath(keywords, glove).fit(train_rows, catalog=catalog_ids),
        "two-tower": TwoTowerRetrievalPath(seed=0).fit(train_rows, catalog=catalog_ids),
    }


serving = fit_six(rows)  # fit on the FULL train history -- the serving-time paths
print(f"catalog {len(catalog_ids)} books; {len(serving)} retrieval paths fit on the full train history")
print(f"readers with a val positive to score: {sum(1 for r in val_by if val_by[r] - seen_by[r])}")

catalog 2000 books; 6 retrieval paths fit on the full train history
readers with a val positive to score: 500


## Stage 1: retrieve a pool, then rank it -- and the recall ceiling

A two-stage recommender splits the work. **Retrieval** is cheap and high-recall: ask each path for its
top `pool` candidates and union them into one pool per reader. **Ranking** is expensive and precise: a
good model re-scores only those `pool` items, not the whole catalog. The reranker lives in stage two,
so it can only ever *re-order the pool* -- if retrieval never proposed a relevant book, no reranker can
surface it.

That gives a hard ceiling. **Pool recall** -- the fraction of scorable readers whose pool contains at
least one of their relevant books -- caps hit@k for *any* reranker. We pin `pool=50`: measured it is
**0.772** (vs **0.716** at pool 30), so a slightly larger pool lifts the ceiling for almost no cost.

In [2]:
# Build one reader's candidate pool: the union of the six paths' top-POOL calibrated candidates.
reader = next(r for r in sorted(val_by) if r not in cold_readers and val_by[r] - seen_by[r])
pool_ids = set()
for name in PATH_NAMES:
    for cand in serving[name].retrieve(reader, {"seen": seen_by[reader]}, POOL):
        pool_ids.add(cand.item_id)
print(f"reader {reader}: six paths propose a merged pool of {len(pool_ids)} books (<= 6 x {POOL})")


def pool_recall(paths, pool):
    # Any-relevant-in-pool recall over the val cohort: the ceiling any reranker is capped at.
    hits = []
    for r in sorted(val_by):
        relevant = val_by[r] - seen_by[r]
        if r in cold_readers or not relevant:
            continue
        merged = set()
        for name in PATH_NAMES:
            for cand in paths[name].retrieve(r, {"seen": seen_by[r]}, pool):
                merged.add(cand.item_id)
        hits.append(1.0 if merged & relevant else 0.0)
    return float(np.mean(hits))


ceiling = pool_recall(serving, POOL)
print(f"pool recall ceiling @pool{POOL}: {ceiling:.3f}   (measured 0.716 @pool 30 -- larger pool, higher ceiling)")
print("No reranker can beat this: it only re-orders the pool retrieval already proposed.")

reader 1: six paths propose a merged pool of 218 books (<= 6 x 50)


pool recall ceiling @pool50: 0.772   (measured 0.716 @pool 30 -- larger pool, higher ceiling)
No reranker can beat this: it only re-orders the pool retrieval already proposed.


## The baseline we must beat: the pool's own fixed score-order

`rank.py` has ranked the pool the same way since Unit 1 -- score descending, ties by id. A natural
stage-two baseline is to blend all six paths' calibrated scores with **equal weights** and read that
fixed order over the `pool=50` candidates. That is the bar a learned reranker has to clear: the
same pool, ordered by a formula we did not learn. We also record the two-tower alone -- the best single
path (hit@10 0.340).

In [3]:
def board(path, pool):
    # hit@10 + catalog coverage for a single path via the blended scoreboard (reproduces its hit@10).
    return run_blended_scoreboard(
        {path.name: path}, interactions_path,
        catalog_ids=catalog_ids, k=K, pool=pool, cold_readers=cold_readers,
    )


six_way = run_blended_scoreboard(
    {name: serving[name] for name in PATH_NAMES}, interactions_path,
    catalog_ids=catalog_ids, k=K, pool=POOL, cold_readers=cold_readers,
)
two_tower = board(serving["two-tower"], K)
print(f"{'baseline':>34} {'hit@10':>8} {'coverage':>10}")
print(f"{'two-tower (U8), best single path':>34} {two_tower.hit_rate_at_k:>8.3f} {two_tower.catalog_coverage:>10.3f}")
print(f"{'6-way equal-weight score-order @pool50':>34} {six_way.hit_rate_at_k:>8.3f} {six_way.catalog_coverage:>10.3f}")
print()
print(f"The bar: the fixed 6-way score-order over the SAME pool the reranker re-orders "
      f"(hit {six_way.hit_rate_at_k:.3f}).")

                          baseline   hit@10   coverage
  two-tower (U8), best single path    0.340      0.177
6-way equal-weight score-order @pool50    0.348      0.216

The bar: the fixed 6-way score-order over the SAME pool the reranker re-orders (hit 0.348).


## Stage 2, the features: a per-candidate ranking vector

A learned ranker scores each pooled candidate from a **feature vector**. Ours is **17-dimensional**,
in four blocks (the reranker persists this exact ordering so a stored vector is interpretable):

- **6 path scores** -- each path's per-list calibrated `[0,1]` score for this candidate (an absent path
  contributes `0.0`). These come from the **pre-blend per-path lists**, *not* the blend's summed score:
  the reranker needs to see each signal separately.
- **6 presence flags** -- a `0/1` per path recording whether it proposed this candidate at all.
- **`n_paths`** -- how many of the six paths surfaced it (agreement is a signal).
- **4 content features** -- `genre_frac` and `genre_cos` (genre affinity to the reader's history),
  `author_frac` (how much of the reader's history shares this book's author), and `log_pop`
  (log-popularity). These describe the *book-reader fit* no single path's score carries.

`feature_names()` returns that ordering.

In [4]:
names = feature_names()
print(f"{len(names)}-dim ranking feature vector, by block:")
print(f"  path scores (6):   {names[0:6]}")
print(f"  presence flags(6): {names[6:12]}")
print(f"  meta (1):          {names[12:13]}")
print(f"  content (4):       {names[13:17]}")

17-dim ranking feature vector, by block:
  path scores (6):   ['score:popularity', 'score:lexical', 'score:item-item', 'score:mf', 'score:semantic', 'score:two-tower']
  presence flags(6): ['in:popularity', 'in:lexical', 'in:item-item', 'in:mf', 'in:semantic', 'in:two-tower']
  meta (1):          ['n_paths']
  content (4):       ['genre_frac', 'genre_cos', 'author_frac', 'log_pop']


## The leakage trap (watch it tank)

Here is the mistake almost everyone makes first. Train the reranker the naive way: use each reader's
train positives as the **positive labels**, and build the features from the paths fit on that *same*
train. It sounds right -- and it is a **leak**. The paths have already *seen* those positives, so they
score them inflatedly; the reranker learns "high score => positive" from in-train scores that
validation candidates never exhibit (distribution shift). The shipped `NeuralRerankerPath` reproduces
this anti-pattern with `leaky=True`. Watch it score **below plain score-order**.

In [5]:
leaky = NeuralRerankerPath(
    serving, lambda subset: serving, catalog, pool=POOL, seed=0, leaky=True,
).fit(rows)
leaky_board = board(leaky, K)
print(f"leaky (naive same-train) reranker hit@10: {leaky_board.hit_rate_at_k:.3f}")
print(f"  ... vs 6-way score-order {six_way.hit_rate_at_k:.3f} and two-tower {two_tower.hit_rate_at_k:.3f}")
print("It TANKED -- worse than the fixed order it was supposed to improve. The features memorized the labels.")

leaky (naive same-train) reranker hit@10: 0.288
  ... vs 6-way score-order 0.348 and two-tower 0.340
It TANKED -- worse than the fixed order it was supposed to improve. The features memorized the labels.


## The fix: a time-ordered holdout inside train

The labels must come from data the feature-producing paths did **not** see. So we split each reader's
train positives by time: the **latest ~25%** become the reranker's labels, and the **earlier 75%** is
the profile -- the `seen` set the paths are refit on, and the history every content and popularity
statistic is computed from. The held-out 25% supplies labels *only*. (A book a reader read both early and late can sit in the
profile *and* the labels -- see Exercise 3 -- so, to be safe, every training-time input is built from
the profile rows that **exclude every occurrence of a held-out-label item**: no held-out event ever
enters a feature-path or the popularity count.) At serving time we go back to the full-fit paths and
the full-train history. Same feature code, phase-dependent input.

`split_profile_labels` builds that split; we refit the six paths **once** on the 75% profile (the
leakage-safe *training* feature producers).

In [6]:
profile, labels = split_profile_labels(rows, 0.25)
example = next(r for r in sorted(labels) if labels[r] and profile.get(r))
print(f"reader {example}: {len(profile[example])} profile positives (75%, build features) + "
      f"{len(labels[example])} held-out labels (25%, supervise only)")

# Refit the six paths on the 75% profile (drop each reader's held-out label rows). Done ONCE.
profile_rows = [
    row for row in rows
    if not (row["split"] == "train" and row["label"] == 1
            and row["item_id"] in labels.get(row["reader_id"], set()))
]
profile_paths = fit_six(profile_rows)
print(f"refit {len(profile_paths)} paths on the profile rows; held-out labels are invisible to them")

reader 0: 12 profile positives (75%, build features) + 5 held-out labels (25%, supervise only)


refit 6 paths on the profile rows; held-out labels are invisible to them


## Learning to rank: assemble the features once, then try models cheaply

The reranker is a **pointwise** learner: each pooled candidate is one training row, labelled `1` if it
is a held-out positive and `0` otherwise (with ~10 sampled pool negatives per positive), trained under
binary cross-entropy. The simplest ranker is a single `Linear` layer -- the Unit-6 blend's weighted
sum, except the weights are now *learned* instead of hand-set. Add a hidden layer and a ReLU and it is
a small MLP:

```python
import torch

# a single-layer logistic combiner (the learned linear blend)...
linear = torch.nn.Linear(17, 1)

# ...or a two-layer MLP over the same 17 features.
mlp = torch.nn.Sequential(
    torch.nn.Linear(17, 32),
    torch.nn.ReLU(),
    torch.nn.Linear(32, 1),
)
loss_fn = torch.nn.BCEWithLogitsLoss()   # pointwise logistic over pool positives vs sampled negatives
```

`NeuralRerankerPath` wraps exactly this, and splits the work the way the cost dictates. **Assembling
the feature matrix is the expensive step** (one pass over every reader, re-running all six paths);
training the little model on it is under a second. So we `assemble_training_matrix` **once** -- it
builds the leakage-safe `(X, y)` from the profile paths -- and then every model variant (linear, MLP,
an ablation) is a cheap `RerankerModel.fit` over that cached matrix, installed with `set_model`.

In [7]:
path = NeuralRerankerPath(serving, lambda subset: profile_paths, catalog, pool=POOL, seed=0)
# Assemble the leakage-safe training matrix ONCE: features from the 75% profile paths, labels from the
# held-out 25%. This is the ~15 s cost; every model below reuses this cached (X, y).
X, y = path.assemble_training_matrix(rows, train_paths=profile_paths)
print(f"training matrix X {X.shape}: {int(y.sum())} held-out positives + {int((y == 0).sum())} sampled "
      f"pool negatives across {len(y)} rows")

training matrix X (6875, 17): 625 held-out positives + 6250 sampled pool negatives across 6875 rows


In [8]:
# Two models over the SAME cached features: a learned linear combiner, and the two-layer MLP.
linear_model = RerankerModel(seed=0, linear=True).fit(X, y)
mlp_model = RerankerModel(seed=0).fit(X, y)   # all 17 features, 17 -> 32 -> 1 (the shipped default)

path.set_model(linear_model)
linear_board = board(path, K)
path.set_model(mlp_model)
clean_board = board(path, K)
print(f"learned linear combiner  hit@10: {linear_board.hit_rate_at_k:.3f}  cov {linear_board.catalog_coverage:.3f}")
print(f"learned MLP reranker     hit@10: {clean_board.hit_rate_at_k:.3f}  cov {clean_board.catalog_coverage:.3f}")
print(f"linear ~= MLP here (gap {abs(linear_board.hit_rate_at_k - clean_board.hit_rate_at_k):.3f}): "
      "the non-linear combiner adds nothing on this data.")

registry = PathRegistry()
registry.register(path)   # the MLP reranker is now installed as the ordering key
print(f"registered as: {path.artifact_name()}")

learned linear combiner  hit@10: 0.364  cov 0.133
learned MLP reranker     hit@10: 0.346  cov 0.195
linear ~= MLP here (gap 0.018): the non-linear combiner adds nothing on this data.
registered as: reranker-v1


## Read the scoreboard honestly: a tie, not a win

The clean MLP reranker reaches **hit@10 ~0.346**. Put it next to the baselines and be honest: it
**ties** the 6-way score-order (0.348) and roughly **matches** the two-tower (0.340) -- the gaps are
well inside a standard error. A leakage-safe learned reranker over all 17 features does **not** beat a
careful fixed blend on this data. The learned linear combiner is no worse than the MLP (the two sit within a
few hundredths; linear's 0.364 is itself within one standard error of 0.348): the non-linearity buys
nothing here.

Now the other column. Coverage is **~0.195** -- a little above the two-tower (0.177), but far below the
Unit-6 blend's 0.333 (and the linear variant concentrates even harder, ~0.13). A ranker that chases
accuracy concentrates its picks on a smaller slice of the catalog. We report both columns -- never just
the one that flatters the model.

In [9]:
print(f"{'path / ranker':>34} {'hit@10':>8} {'coverage':>10}")
for label, hit, cov in (
    ("two-tower (U8)", two_tower.hit_rate_at_k, two_tower.catalog_coverage),
    ("U6 4-way blend (recorded)", 0.306, 0.333),
    ("U10 hybrid (recorded)", 0.362, 0.192),
    ("6-way score-order @pool50", six_way.hit_rate_at_k, six_way.catalog_coverage),
    ("learned linear combiner", linear_board.hit_rate_at_k, linear_board.catalog_coverage),
    ("learned MLP reranker", clean_board.hit_rate_at_k, clean_board.catalog_coverage),
):
    print(f"{label:>34} {hit:>8.3f} {cov:>10.3f}")
print()
print("Accuracy: the all-feature reranker TIES the 6-way score-order and ~matches the two-tower.")
print("Coverage: edges the two-tower, but a real LOSS vs the U6 blend (0.333).")

                     path / ranker   hit@10   coverage
                    two-tower (U8)    0.340      0.177
         U6 4-way blend (recorded)    0.306      0.333
             U10 hybrid (recorded)    0.362      0.192
         6-way score-order @pool50    0.348      0.216
           learned linear combiner    0.364      0.133
              learned MLP reranker    0.346      0.195

Accuracy: the all-feature reranker TIES the 6-way score-order and ~matches the two-tower.
Coverage: edges the two-tower, but a real LOSS vs the U6 blend (0.333).


## A second, subtler leak: the +0.018 phantom

The time-ordered holdout closed the obvious leak. But an earlier version of this reranker still scored
**0.364** -- and it was a **phantom**. Here is how. The `log_pop` content feature counts how often each
book was read in training. That count was built from every train row whose book sat in the reader's
*profile set*. But a reader can read the same book more than once: an early read lands in the profile,
and the *latest* read becomes a held-out label. The book is then in the profile **and** in the labels
-- so counting "rows whose book is in the profile" also counted the **held-out read itself**. The very
event the reranker was asked to predict was quietly bumping that book's popularity feature.

The fix works at the level of **events**, not item sets: training popularity is counted from the
profile rows that **exclude every occurrence of a held-out-label item** (the same `profile_rows` we
refit the paths on). With that fix the all-feature reranker drops from 0.364 to **0.346** -- the whole
+0.018 "win" over the score-order was leakage. Lesson: "leakage-safe" is a claim about *events*. Check
it event by event.

In [10]:
# Count the repeat-read trap: train rows whose book is BOTH in that reader's profile set and among
# their held-out labels. Counting popularity by profile *membership* would include these rows -- and
# with them the held-out read itself. The shipped reranker counts from profile_rows, which drop them all.
trap_rows = [
    row for row in rows
    if row["split"] == "train" and row["label"] == 1
    and row["item_id"] in labels.get(row["reader_id"], set())
    and row["item_id"] in profile.get(row["reader_id"], set())
]
trap_readers = {row["reader_id"] for row in trap_rows}
print(f"{len(trap_rows)} train rows from {len(trap_readers)} readers sit on a profile-AND-label book")
print(f"rows kept for training popularity: {len(profile_rows)} of {len(rows)} (every label-item row dropped)")
print("recorded: pre-fix all-feature reranker 0.364 -> leakage-safe 0.346 (a +0.018 phantom)")

623 train rows from 181 readers sit on a profile-AND-label book
rows kept for training popularity: 25300 of 27004 (every label-item row dropped)
recorded: pre-fix all-feature reranker 0.364 -> leakage-safe 0.346 (a +0.018 phantom)


## Where the lift actually comes from (the payoff)

So the all-feature reranker only ties a fixed blend. Is there *any* real lift in learning to rank here?
Surely a learned ranker's strength is cleverly *combining the path scores*? We test it directly: train
one reranker on the **content features only** (drop every path score and presence flag) and another on
the **scores only** (drop all content), and score both on `val`.

In [11]:
# Same cached (X, y); a family_mask keeps only the named feature blocks, so each ablation is a
# sub-second RerankerModel.fit installed with set_model -- no re-assembly.
content_model = RerankerModel(seed=0).fit(X, y, feature_mask=family_mask(("content",)))
scores_model = RerankerModel(seed=0).fit(X, y, feature_mask=family_mask(("scores", "presence", "meta")))
path.set_model(content_model)
content_board = board(path, K)
path.set_model(scores_model)
scores_board = board(path, K)
print(f"content-only reranker hit@10: {content_board.hit_rate_at_k:.3f}")
print(f"scores-only  reranker hit@10: {scores_board.hit_rate_at_k:.3f}")
print()
print(f"all-feature  reranker hit@10: {clean_board.hit_rate_at_k:.3f}   6-way score-order: {six_way.hit_rate_at_k:.3f}")
print()
print("Content-only is the BEST reranker and clearly beats the fixed score-order. The")
print("lift is content affinity (genre/author fit to the reader's history), a signal no single path's")
print("score carries. Adding the correlated path scores DILUTES it: all-feature < content-only.")

content-only reranker hit@10: 0.380
scores-only  reranker hit@10: 0.346

all-feature  reranker hit@10: 0.346   6-way score-order: 0.348

Content-only is the BEST reranker and clearly beats the fixed score-order. The
lift is content affinity (genre/author fit to the reader's history), a signal no single path's
score carries. Adding the correlated path scores DILUTES it: all-feature < content-only.


### Reproducible, not random

The reranker is deterministic (design 011 section 184): its training seeds torch's init and the numpy
negative sampling, runs single-threaded with deterministic algorithms, and saves/restores those
process-global knobs. Two seeded fits give **identical top-k rankings** and `allclose` weights -- the
gate we hold it to, never exact-float identity.

In [12]:
# Two seeded fits on the same cached matrix must agree on every weight (the gate) and produce the
# same top-k ranking (what the reader sees). We never demand exact-float identity.
first = RerankerModel(seed=0).fit(X, y)
second = RerankerModel(seed=0).fit(X, y)
all_close = all(np.allclose(a, b) for a, b in zip(first.weights, second.weights))

readers = [r for r in sorted(val_by) if r not in cold_readers and val_by[r] - seen_by[r]][:5]
path.set_model(first)
ranking_a = {r: [c.item_id for c in path.retrieve(r, {"seen": seen_by[r]}, K)] for r in readers}
path.set_model(second)
ranking_b = {r: [c.item_id for c in path.retrieve(r, {"seen": seen_by[r]}, K)] for r in readers}
print(f"allclose on every learned weight: {all_close}")
print(f"identical top-10 for {len(readers)} readers: {ranking_a == ranking_b}")

allclose on every learned weight: True
identical top-10 for 5 readers: True


## Where this goes

- A two-stage **retrieve-then-rank** recommender retrieves a cheap, high-recall **pool** and re-scores
  only that pool with an expensive model. The reranker can only re-order the pool, so **pool recall**
  (~0.77 at pool 50) caps hit@k -- retrieval quality still matters.
- The reranker scores each candidate from a **17-dim feature vector**: the pre-blend per-path
  calibrated scores, presence flags, and content affinity to the reader's history.
- **Two leak modes** are the headline lesson. (1) *Path memorization*: train on labels the
  feature-producing paths have already seen and the reranker **tanks to ~0.29**, below score-order --
  fixed by a **time-ordered holdout inside train**. (2) *Held-out-event inflation*: a repeat-read
  held-out event leaked into `log_pop` and faked a **+0.018** win (0.364) -- fixed by counting training
  popularity from the label-item-excluded profile rows. Leakage-safety is checked event by event.
- Read it honestly: with both leaks closed, the all-feature reranker (~0.346) **ties** the 6-way
  score-order (0.348) and ~matches the two-tower (0.340); coverage (~0.195) edges the two-tower but is
  well below the Unit-6 blend (0.333). The genuine lift is **content features**: content-only (~0.380)
  beats the score-order, the all-feature model dilutes it, and **linear ~= MLP**.

### The bridge ahead

**Unit 12** adds **sequence features** -- the *order* of a reader's history, not a bag of it -- as new
ranking inputs (more content a single path's score cannot see). **Unit 13** revisits the fairness of a
*learned* ranker and the coverage cost we just measured. **Unit 14**, the capstone, wires the whole
stack end to end: retrieve a pool, blend it, and rerank it.